# Local (pandas) baseline - 10 years (2015-2024)

**IST3134 Big Data Group Assignment** - single-machine baseline.

Runs the flight-delay analysis on **10 years (2015-2024)** of US BTS On-Time Performance data using **pandas on one machine** - the non-big-data side of the comparison against PySpark. Records **runtime** and **peak memory** for the 1/5/10-year comparison.

At the end it writes a **single HTML report** (all tables + charts + performance) to `Output/10year/` so you can view results without scrolling through code.

> "Delayed" = arrival delay >= 15 minutes.

## 1. Setup & config

In [1]:
import os, time, zipfile, tracemalloc
import pandas as pd

RAW_DIR = os.path.join("..", "Data", "Raw")
DURATION_LABEL = "10 years (2015-2024)"
TAG = "10year"
FILES = [
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2015_1.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2015_2.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2015_3.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2015_4.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2015_5.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2015_6.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2015_7.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2015_8.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2015_9.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2015_10.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2015_11.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2015_12.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2016_1.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2016_2.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2016_3.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2016_4.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2016_5.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2016_6.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2016_7.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2016_8.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2016_9.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2016_10.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2016_11.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2016_12.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2017_1.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2017_2.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2017_3.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2017_4.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2017_5.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2017_6.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2017_7.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2017_8.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2017_9.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2017_10.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2017_11.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2017_12.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2018_1.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2018_2.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2018_3.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2018_4.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2018_5.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2018_6.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2018_7.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2018_8.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2018_9.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2018_10.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2018_11.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2018_12.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2019_1.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2019_2.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2019_3.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2019_4.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2019_5.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2019_6.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2019_7.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2019_8.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2019_9.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2019_10.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2019_11.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2019_12.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2020_1.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2020_2.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2020_3.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2020_4.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2020_5.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2020_6.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2020_7.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2020_8.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2020_9.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2020_10.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2020_11.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2020_12.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2021_1.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2021_2.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2021_3.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2021_4.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2021_5.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2021_6.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2021_7.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2021_8.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2021_9.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2021_10.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2021_11.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2021_12.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2022_1.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2022_2.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2022_3.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2022_4.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2022_5.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2022_6.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2022_7.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2022_8.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2022_9.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2022_10.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2022_11.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2022_12.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2023_1.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2023_2.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2023_3.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2023_4.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2023_5.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2023_6.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2023_7.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2023_8.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2023_9.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2023_10.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2023_11.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2023_12.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_1.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_2.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_3.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_4.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_5.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_6.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_7.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_8.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_9.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_10.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_11.zip",
"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_12.zip"
]
USECOLS = ["Year", "Month", "DayOfWeek", "Reporting_Airline", "Origin", "Dest", "CRSDepTime", "DepDelay", "ArrDelay", "ArrDelayMinutes", "Cancelled", "Distance", "CarrierDelay", "WeatherDelay", "NASDelay", "SecurityDelay", "LateAircraftDelay"]
print("Duration:", DURATION_LABEL)
print("Monthly files:", len(FILES))

Duration: 10 years (2015-2024)
Monthly files: 120


## 2. Load
Only needed columns are read (`usecols`) to limit memory. pandas holds everything in RAM, so large durations will strain or fail - that is part of the comparison.

In [2]:
def read_month(zip_path):
    """Read the single on-time CSV inside a BTS monthly zip."""
    with zipfile.ZipFile(zip_path) as z:
        csv_name = [n for n in z.namelist() if n.lower().endswith(".csv")][0]
        with z.open(csv_name) as f:
            return pd.read_csv(f, usecols=USECOLS, low_memory=False)

tracemalloc.start()
t0 = time.perf_counter()
frames = []
for fn in FILES:
    path = os.path.join(RAW_DIR, fn)
    if not os.path.exists(path):
        print("MISSING:", fn, "-> download into Data/Raw"); continue
    frames.append(read_month(path))
df = pd.concat(frames, ignore_index=True)
load_secs = time.perf_counter() - t0
print(f"Loaded {len(df):,} rows in {load_secs:,.1f}s")

Loaded 63,079,426 rows in 248.0s


## 3. Clean
Flag delayed flights; derive departure hour and route key.

In [3]:
df = df[df["Cancelled"] != 1].copy()
df["delayed"] = (df["ArrDelay"] >= 15).astype(int)
df["dep_hour"] = (df["CRSDepTime"] // 100).clip(0, 23)
df["route"] = df["Origin"] + "-" + df["Dest"]
print("Flights after cleaning:", f"{len(df):,}")

Flights after cleaning: 61,839,886


## 4. Analysis
Five result tables, one per sub-question.

In [4]:
# 4a. Q1 by carrier
by_carrier = (df.groupby("Reporting_Airline")
    .agg(flights=("delayed","size"), avg_arr_delay=("ArrDelay","mean"), pct_delayed=("delayed","mean"))
    .sort_values("pct_delayed", ascending=False))
by_carrier["pct_delayed"] *= 100
by_carrier.head(15)

,flights,avg_arr_delay,pct_delayed
Reporting_Airline,,,
F9,1287500,11.258745,26.041476
B6,2528111,10.908267,25.940752
G4,739094,11.810566,24.602960
VX,217006,7.390080,23.403040
NK,1837073,8.036908,22.434383
AA,8334724,6.792022,20.218930
EV,1739339,7.136827,19.593650
WN,12526239,3.931050,19.291385
YV,824697,7.978104,19.272654


In [5]:
# 4b. Q2 worst airports (origin), min 100,000 flights
by_airport = (df.groupby("Origin")
    .agg(flights=("delayed","size"), avg_arr_delay=("ArrDelay","mean"), pct_delayed=("delayed","mean")))
by_airport = by_airport[by_airport["flights"] >= 100000]
by_airport["pct_delayed"] *= 100
by_airport.sort_values("pct_delayed", ascending=False).head(15)

,flights,avg_arr_delay,pct_delayed
Origin,,,
SCK,4653,20.642151,34.128519
HTS,3720,21.605987,32.123656
MMH,1091,24.024748,31.805683
HGR,1371,21.145681,31.291028
LCK,7016,19.034291,30.416192
USA,6707,18.257139,30.013419
BLV,7439,18.323581,28.565667
PSM,2389,14.003357,28.254500
HYA,1168,19.004288,27.739726


In [6]:
# 4c. Q3 worst routes, min 5,000 flights
by_route = (df.groupby("route")
    .agg(flights=("delayed","size"), avg_arr_delay=("ArrDelay","mean"), pct_delayed=("delayed","mean")))
by_route = by_route[by_route["flights"] >= 5000]
by_route["pct_delayed"] *= 100
by_route.sort_values("pct_delayed", ascending=False).head(15)

,flights,avg_arr_delay,pct_delayed
route,,,
AUS-HNL,582,47.551724,52.577320
LEX-EWR,688,50.292398,51.308140
EWR-FWA,607,32.099010,47.611203
XNA-EWR,997,29.253776,42.527583
MRY-LAS,726,31.190083,42.148760
LAX-CHS,536,23.930970,41.231343
MIA-PDX,501,24.253012,41.117764
GSP-FLL,673,34.600000,40.861813
EWR-OKC,702,20.943804,40.455840


In [7]:
# 4d. Q4 by time of day (scheduled dep hour)
by_hour = (df.groupby("dep_hour")
    .agg(flights=("delayed","size"), avg_arr_delay=("ArrDelay","mean"), pct_delayed=("delayed","mean")))
by_hour["pct_delayed"] *= 100
by_hour

,flights,avg_arr_delay,pct_delayed
dep_hour,,,
0,133255,1.481086,16.982477
1,48203,3.743607,17.536253
2,15619,7.155322,22.197324
3,10818,6.914090,23.054169
4,6016,5.344161,19.847074
5,1408940,-3.054534,8.060599
6,4343311,-2.681472,9.170584
7,4226176,-1.560875,11.291887
8,4202304,-0.809626,12.808069


In [8]:
# 4e. Q5 cause mix by year (share of delay minutes)
causes = ["CarrierDelay","WeatherDelay","NASDelay","SecurityDelay","LateAircraftDelay"]
cause_by_year = df.groupby("Year")[causes].sum()
cause_mix = cause_by_year.div(cause_by_year.sum(axis=1), axis=0) * 100
cause_mix

,CarrierDelay,WeatherDelay,NASDelay,SecurityDelay,LateAircraftDelay
Year,,,,,
2015,32.198491,4.948349,22.881620,0.129262,39.842278
2016,32.636958,4.354331,23.676745,0.136883,39.195083
2017,31.172587,4.253887,25.070902,0.143080,39.359545
2018,30.047658,5.617042,24.557169,0.144514,39.633616
2019,30.611164,5.508934,24.032664,0.139206,39.708033
2020,41.844845,7.039496,21.670710,0.234427,29.210521
2021,40.828598,6.875509,16.708520,0.339272,35.248101
2022,39.783631,5.552372,16.803782,0.211822,37.648392
2023,36.429214,5.234524,18.088158,0.215968,40.032137


## 5. Performance (for the comparison table)
Total runtime and peak memory - record for 1/5/10 years and compare against Spark.

In [9]:
total_secs = time.perf_counter() - t0
cur, peak = tracemalloc.get_traced_memory(); tracemalloc.stop()
print(f"Duration analysed : {DURATION_LABEL}")
print(f"Rows              : {len(df):,}")
print(f"Load time         : {load_secs:,.1f} s")
print(f"Total runtime     : {total_secs:,.1f} s")
print(f"Peak memory       : {peak/1e9:,.2f} GB")

Duration analysed : 10 years (2015-2024)
Rows              : 61,839,886
Load time         : 248.0 s
Total runtime     : 479.0 s
Peak memory       : 38.44 GB


## 6. Charts
One chart per sub-question (also embedded in the HTML report below).

In [ ]:
# 6. Build charts (one per sub-question)
import matplotlib
import matplotlib.pyplot as plt
BLUE = "#2563eb"
figs = {}

# Readable lookups. Airport and city names come straight from the dataset
# (OriginCityName / DestCityName), airline names are a fixed code lookup because
# the raw data only carries the 2 letter carrier code.
city = dict(zip(df["Origin"], df["OriginCityName"]))
city.update(dict(zip(df["Dest"], df["DestCityName"])))
def city_full(code):  return city.get(code, code)
def city_short(code): return city.get(code, code).split(",")[0]

airline = {"F9":"Frontier Airlines","B6":"JetBlue Airways","G4":"Allegiant Air",
"VX":"Virgin America","NK":"Spirit Airlines","AA":"American Airlines","EV":"ExpressJet",
"WN":"Southwest Airlines","YV":"Mesa Airlines","US":"US Airways","UA":"United Airlines",
"MQ":"Envoy Air","OH":"PSA Airlines","AS":"Alaska Airlines","OO":"SkyWest Airlines",
"9E":"Endeavor Air","YX":"Republic Airways","HA":"Hawaiian Airlines","DL":"Delta Air Lines",
"QX":"Horizon Air"}

def hbar(labels, vals, title, key, figsize=(9,5)):
    fig, ax = plt.subplots(figsize=figsize)
    y = range(len(labels))
    bars = ax.barh(list(y), list(vals), color=BLUE)
    ax.set_yticks(list(y)); ax.set_yticklabels(labels)
    ax.invert_yaxis()
    ax.set_title(title); ax.set_xlabel("% of flights delayed (>=15 min)")
    ax.set_xlim(0, max(vals) * 1.15)
    ax.bar_label(bars, fmt="%.1f%%", padding=3, fontsize=9)
    fig.tight_layout(); figs[key] = fig

# Q1 airlines - top 15 by % delayed, readable airline names
top = by_carrier.sort_values("pct_delayed", ascending=False).head(15)
hbar([airline.get(c, c) for c in top.index], top["pct_delayed"],
     "Q1 - Worst airlines by % delayed", "Q1_carrier", (9, 5.2))

# Q2 major airports - readable "City, ST (CODE)"
top = by_airport.sort_values("pct_delayed", ascending=False).head(12)
hbar([f"{city_full(c)} ({c})" for c in top.index], top["pct_delayed"],
     "Q2 - Worst major airports by % delayed", "Q2_airport")

# Q3 major routes - readable "City to City"
top = by_route.sort_values("pct_delayed", ascending=False).head(12)
def route_label(r):
    o, d = r.split("-"); return f"{city_short(o)} to {city_short(d)}"
hbar([route_label(r) for r in top.index], top["pct_delayed"],
     "Q3 - Worst major routes by % delayed", "Q3_route")

# Q4 time of day - line with periodic value labels
fig, ax = plt.subplots(figsize=(9, 4.6))
ax.plot(by_hour.index, by_hour["pct_delayed"], marker="o", color=BLUE)
ax.set_title("Q4 - Delay rate by departure hour")
ax.set_xlabel("Scheduled departure hour"); ax.set_ylabel("% of flights delayed (>=15 min)")
ax.set_xticks(range(0, 24, 2)); ax.grid(True, alpha=0.3)
for x, yv in zip(by_hour.index, by_hour["pct_delayed"]):
    if x % 3 == 0:
        ax.annotate(f"{yv:.1f}%", (x, yv), textcoords="offset points",
                    xytext=(0, 7), ha="center", fontsize=8)
fig.tight_layout(); figs["Q4_hour"] = fig

# Q5 cause mix over time
fig, ax = plt.subplots(figsize=(9, 4.6))
cause_mix.plot.area(ax=ax, cmap="tab10", alpha=0.9)
ax.set_title("Q5 - Delay cause mix by year"); ax.set_xlabel("Year"); ax.set_ylabel("Share of delay minutes (%)")
ax.set_ylim(0, 100); ax.legend(loc="lower center", ncol=5, fontsize=7)
fig.tight_layout(); figs["Q5_causemix"] = fig

print("Built", len(figs), "charts")


## 7. Output report
Writes everything to `Output/10year/report_10year.html` - open that one file to see all tables and charts, no code.

In [11]:
# 7. Write a single self-contained HTML report to Output/<duration>/
import os, io, base64

OUTPUT_DIR = os.path.join("..", "Output", TAG)
os.makedirs(OUTPUT_DIR, exist_ok=True)

def embed_and_save(name, fig):
    fig.savefig(os.path.join(OUTPUT_DIR, name + ".png"), dpi=120, bbox_inches="tight")
    buf = io.BytesIO(); fig.savefig(buf, format="png", dpi=120, bbox_inches="tight"); buf.seek(0)
    return base64.b64encode(buf.read()).decode()

sections = [
    ("Q1 - Worst airlines (by % delayed)", by_carrier.sort_values("pct_delayed", ascending=False).head(15).round(1), "Q1_carrier"),
    ("Q2 - Worst origin airports", by_airport.sort_values("pct_delayed", ascending=False).head(15).round(1), "Q2_airport"),
    ("Q3 - Worst routes", by_route.sort_values("pct_delayed", ascending=False).head(15).round(1), "Q3_route"),
    ("Q4 - Delay by time of day", by_hour.round(1), "Q4_hour"),
    ("Q5 - Cause mix by year (%)", cause_mix.round(1), "Q5_causemix"),
]

css = """<style>
body{font-family:Segoe UI,Arial,sans-serif;margin:32px;color:#1e293b;max-width:1000px}
h1{color:#1F4E79} h2{color:#2563eb;border-bottom:2px solid #e2e8f0;padding-bottom:4px;margin-top:32px}
table{border-collapse:collapse;margin:12px 0;font-size:13px} th,td{border:1px solid #cbd5e1;padding:4px 8px;text-align:right}
th{background:#f1f5f9} td:first-child,th:first-child{text-align:left}
.perf{background:#f8fafc;border:1px solid #e2e8f0;border-radius:8px;padding:12px 16px;margin:12px 0}
img{max-width:680px;display:block;margin:8px 0}
</style>"""

html = [f"<html><head><meta charset='utf-8'>{css}</head><body>"]
html.append(f"<h1>Flight Delay Analysis - Local (pandas) - {DURATION_LABEL}</h1>")
html.append("<div class='perf'><b>Performance summary</b><br>"
            f"Rows analysed: {len(df):,}<br>"
            f"Load time: {load_secs:,.1f} s<br>"
            f"Total runtime: {total_secs:,.1f} s<br>"
            f"Peak memory: {peak/1e9:,.2f} GB</div>")

for title, table, figname in sections:
    b64 = embed_and_save(figname, figs[figname])
    html.append(f"<h2>{title}</h2>")
    html.append(f"<img src='data:image/png;base64,{b64}'>")
    html.append(table.to_html())

html.append("</body></html>")
report_path = os.path.join(OUTPUT_DIR, f"report_{TAG}.html")
with open(report_path, "w", encoding="utf-8") as f:
    f.write("\n".join(html))
print("Report written to:", report_path)
print("Open that file to see all tables + charts without any code.")

Report written to: ..\Output\10year\report_10year.html
Open that file to see all tables + charts without any code.


## 8. Save result tables as CSV (optional)

In [ ]:
out = os.path.join("..", "Data", "Cleaned"); os.makedirs(out, exist_ok=True)
by_carrier.to_csv(os.path.join(out, f"pandas_by_carrier_{TAG}.csv"))
by_airport.sort_values("pct_delayed", ascending=False).to_csv(os.path.join(out, f"pandas_by_airport_{TAG}.csv"))
by_route.sort_values("pct_delayed", ascending=False).to_csv(os.path.join(out, f"pandas_by_route_{TAG}.csv"))
by_hour.to_csv(os.path.join(out, f"pandas_by_hour_{TAG}.csv"))
cause_mix.to_csv(os.path.join(out, f"pandas_cause_mix_{TAG}.csv"))
print("Saved CSVs to Data/Cleaned")

Saved CSVs to Data/Cleaned
